### One-time environment provisioning
##### Run manually once per environment (dev/staging/prod) -- not `%run` from pipeline notebooks. Superseded once Terraform/DABs IaC lands; keep as the manual fallback until then.

In [0]:
%run ./config

In [ ]:
def create_directory(path: str) -> None:
    try:
        dbutils.fs.ls(path)
        print("Directory verified via ls.")
    except Exception:
        dbutils.fs.mkdirs(path)
        
    return

In [ ]:
# The catalog's managed-location folder must exist before the catalog can use it (safe to re-run).
create_directory(f"abfss://landing@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/managed/{CATALOG_NAME}/")

In [ ]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}
    MANAGED LOCATION 'abfss://landing@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/managed/{CATALOG_NAME}/'
""")

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")

In [ ]:
spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

In [ ]:
# Landing volume: where raw GTFS static drops land.
spark.sql(f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}
    LOCATION 'abfss://landing@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/{CATALOG_NAME}/'
""")

In [ ]:
create_directory(f"abfss://checkpoints@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/{CATALOG_NAME}/")    

In [ ]:
# Separate container from landing, so the checkpoint dir never matches a pathGlobFilter.
spark.sql(f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{CHECKPOINT_VOLUME_NAME}
    LOCATION 'abfss://checkpoints@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/{CATALOG_NAME}/'
""")

In [ ]:
print(f"Provisioned {CATALOG_NAME}.{SCHEMA_NAME} with volumes: {VOLUME_NAME}, {CHECKPOINT_VOLUME_NAME}")